In [ ]:
# Jack's Birthday Hash

# we need to find number of unique secrets so that after hashing each of them, we will get same value as value of Jack in 50% probability
# after n tries we will get same hash value in 1-(1-1/2^11)^n
# and it is equal to 0.5=1-(1-1/2^11)^n
# (1-1/2^11)^n=0.5
# n=1419.2 and we will get more than 50% if n=1420

In [ ]:
# Jack's Birthday Confusion

# if we already generated different hash values n times then probability that new hash will be equal to at least one of them is 1-(1-1/2^11)^n
# probability that during generation of first n hashes we will have no collision is p(n) then p(n)=p(n-1)(1-1/2^11)^(n-1)=(1-1/2^11)^(n(n-1)/2)
# probability that we have no collision is 1-(1-1/2^11)^(n(n-1)/2)=0.75
# n=75.84 hence after 76 trials we will have more than 75% probability

In [9]:
# Collider

from pwn import * # pip install pwntools
import hashlib
import json

# we need to find string so that its md5 hash is same as one of stored hashes 
# also we can add new hash to the stored hashes
# so it is suficient to find two any messages which have same md5 hash
# which I found here https://www.mscs.dal.ca/~selinger/md5collision/

r = remote("socket.cryptohack.org", 13389)
print(r.readline())

def json_send(hsh={}):
    if len(hsh) != 0:
        request = json.dumps(hsh).encode()
        r.sendline(request)
    line = r.readline()
    return json.loads(line.decode())

data1 = bytes.fromhex("d131dd02c5e6eec4693d9a0698aff95c2fcab58712467eab4004583eb8fb7f8955ad340609f4b30283e488832571415a085125e8f7cdc99fd91dbdf280373c5bd8823e3156348f5bae6dacd436c919c6dd53e2b487da03fd02396306d248cda0e99f33420f577ee8ce54b67080a80d1ec69821bcb6a8839396f9652b6ff72a70")

data2 = bytes.fromhex("d131dd02c5e6eec4693d9a0698aff95c2fcab50712467eab4004583eb8fb7f8955ad340609f4b30283e4888325f1415a085125e8f7cdc99fd91dbd7280373c5bd8823e3156348f5bae6dacd436c919c6dd53e23487da03fd02396306d248cda0e99f33420f577ee8ce54b67080280d1ec69821bcb6a8839396f965ab6ff72a70")

assert hashlib.md5(data1).hexdigest() == hashlib.md5(data2).hexdigest()

print(json_send({"document": data1.hex()})["success"])
print(json_send({"document": data2.hex()})["error"])

[x] Opening connection to socket.cryptohack.org on port 13389
[x] Opening connection to socket.cryptohack.org on port 13389: Trying 134.122.111.232
[+] Opening connection to socket.cryptohack.org on port 13389: Done
b'Give me a document to store\n'
Document 79054025255fb1a26e4bc422aef54eb4 added to system
Document system crash, leaking flag: crypto{m0re_th4n_ju5t_p1g30nh0le_pr1nc1ple}


In [ ]:
# PriMeD5

from pwn import * # pip install pwntools
from Crypto.Hash import MD5
from Crypto.Util.number import long_to_bytes, isPrime
import json

r = remote("socket.cryptohack.org", 13392)
print(r.readline())

def json_send(hsh={}):
    if len(hsh) != 0:
        request = json.dumps(hsh).encode()
        r.sendline(request)
    line = r.readline()
    return json.loads(line.decode())

# we need to find prime and composite numbers so that their md5 hash matches
# so we can sign using prime number and check using composite number and provide one of its composite factors as "a" parameter to get gcd>1 
#TODO

assert isPrime(997)
MD5.new(long_to_bytes(997))
signature = json_send({"option": "sign", "prime" : 997})["signature"]
json_send({"option": "check", "prime" : 997, "signature" : signature, "a" : 5})

[x] Opening connection to socket.cryptohack.org on port 13392
[x] Opening connection to socket.cryptohack.org on port 13392: Trying 134.122.111.232
[+] Opening connection to socket.cryptohack.org on port 13392: Done
b'Primality checking is expensive so I made a service that signs primes, allowing anyone to quickly check if a number is prime\n'


{'msg': 'Valid signature. First byte of flag: c'}

In [35]:
# Merkle Trees

import ast
from hashlib import sha256
from Crypto.Util.number import long_to_bytes

def merge_nodes(a, b):
    return sha256(a+b).digest()

binary_repr = b''

with open('MerkleTrees/output.txt', 'r') as f:
    lines = f.readlines()
    for line in lines:
        [a,b,c,d, root] = ast.literal_eval(line.strip())
        left = merge_nodes(bytes.fromhex(a), bytes.fromhex(b))
        right = merge_nodes(bytes.fromhex(c), bytes.fromhex(d))
        root_unbiased = merge_nodes(left, right)
        if root_unbiased.hex() == root:
             binary_repr += b'1'
        else:
             binary_repr += b'0'

print(binary_repr)
print(long_to_bytes(int(binary_repr, 2)))

b'110001101110010011110010111000001110100011011110111101101010101010111110110000101110010011001010101111101010010001100110110000101100100011110010101111101000110011011110111001001011111010100110011010001110000011011000110100101101110011100110101111101100011011010000011010001101100011011000111001101111101'
b'crypto{U_are_R3ady_For_S4plins_ch4lls}'
